# Asnya Research · Hypothesis to evidence
A reproducible daily-price strategy experiment. Synthetic data by default; no market feed.

**Question:** does a simple long/cash trend rule outperform buy-and-hold after costs on untouched chronological test data?

Change parameters, then restart the kernel and run all cells. Repeatedly examining the test split turns it into development data; reserve new data for further claims. Synthetic results establish software behavior, not a profitable edge.


In [ ]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
CSV_PATH = None  # Optional CSV with date,close, adjusted daily closes in ascending order.
TRAIN_FRACTION = 2 / 3
COST_BPS = 5
CANDIDATES = [(10, 40), (20, 60), (40, 120)]
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)


## Engine and execution contract
The following engine is embedded from `asnya_research.py` so this notebook also runs standalone in Colab. Edit the Python source and regenerate the notebook for repository changes.

A signal computed at close t applies to the return t→t+1. Holdings are 100% long or cash. One-way cost in basis points is charged per position change. Each evaluation starts in cash; buy-and-hold pays the same entry cost. Warmup is excluded, cash earns zero, risk-free rate is zero, and annualization assumes 252 daily trading observations. No slippage beyond the configured cost, dividends, leverage, taxes or terminal liquidation cost. Supplied prices should account for corporate actions; the engine does not repair exchange calendars.


In [ ]:
"""Daily long/cash research baseline. No live data or broker execution."""
import hashlib
import numpy as np
import pandas as pd

ENGINE_VERSION = 'research-0.1'

def validate_prices(frame):
    if not {'date', 'close'}.issubset(frame.columns):
        raise ValueError('Expected date,close columns')
    data = frame[['date', 'close']].copy()
    data['date'] = pd.to_datetime(data['date'], format='%Y-%m-%d', errors='raise')
    data['close'] = pd.to_numeric(data['close'], errors='raise')
    if len(data) < 260 or len(data) > 20000:
        raise ValueError('Expected 260–20,000 daily observations')
    if data.isna().any().any() or not np.isfinite(data['close']).all() or (data['close'] <= 0).any():
        raise ValueError('Prices must be finite and positive; no missing values')
    if data['date'].duplicated().any() or not data['date'].is_monotonic_increasing:
        raise ValueError('Dates must be unique and ascending')
    return data.reset_index(drop=True)

def synthetic_prices(seed=42, observations=756):
    rng = np.random.default_rng(seed)
    returns = rng.normal(0.0004, 0.012, observations)
    return validate_prices(pd.DataFrame({'date': pd.bdate_range('2023-01-02', periods=observations),
                                        'close': 100 * np.exp(np.cumsum(returns))}))

def fingerprint(data):
    return hashlib.sha256(data.to_csv(index=False, date_format='%Y-%m-%d').encode()).hexdigest()

def backtest(data, fast=20, slow=60, cost_bps=5, start=None):
    data = validate_prices(data)
    if not isinstance(fast, int) or not isinstance(slow, int) or not 2 <= fast < slow <= 250:
        raise ValueError('Require integer windows: 2 <= fast < slow <= 250')
    if not np.isfinite(cost_bps) or not 0 <= cost_bps <= 200:
        raise ValueError('Cost must be 0–200 bps')
    start = slow if start is None else start
    if not isinstance(start, int) or not slow <= start < len(data) - 1:
        raise ValueError('Invalid evaluation start')
    close = data['close']
    signal = (close.rolling(fast).mean() > close.rolling(slow).mean()).astype(float)
    # Signal at previous close determines exposure to today's close-to-close return.
    position = signal.shift(1).fillna(0).iloc[start:].copy()
    turnover = position.diff().abs()
    turnover.iloc[0] = position.iloc[0]  # Evaluation starts in cash, including held-out test.
    returns = close.pct_change().iloc[start:]
    net = position * returns - turnover * cost_bps / 10000
    benchmark = returns.copy()
    benchmark.iloc[0] -= cost_bps / 10000
    equity = np.r_[100., 100 * (1 + net).cumprod().to_numpy()]
    benchmark_equity = np.r_[100., 100 * (1 + benchmark).cumprod().to_numpy()]
    out = data.iloc[start-1:].copy().reset_index(drop=True)
    out['position'] = np.r_[0., position.to_numpy()]
    out['return'] = np.r_[0., net.to_numpy()]
    out['equity'] = equity
    out['benchmark'] = benchmark_equity
    out['drawdown'] = equity / np.maximum.accumulate(equity) - 1
    sd = float(net.std(ddof=1))
    metrics = {'total_return': float(equity[-1]/100-1),
               'annualized_return': float((equity[-1]/100)**(252/len(net))-1),
               'sharpe': float(net.mean()/sd*np.sqrt(252)) if sd > 0 else 0.,
               'max_drawdown': float(out['drawdown'].min()),
               'position_changes': int(turnover.sum()), 'evaluated_returns': len(net)}
    return out, metrics

def select_on_training(data, split, candidates, cost_bps=5):
    """Reads only rows before split. Shared warmup makes candidates comparable."""
    if not isinstance(split, int) or not 260 <= split <= len(data)-60:
        raise ValueError('Need >=260 training rows and >=60 test rows')
    train = data.iloc[:split].copy()
    common_start = max(slow for _, slow in candidates)
    records = []
    for fast, slow in candidates:
        _, metrics = backtest(train, fast, slow, cost_bps, start=common_start)
        records.append({'fast': fast, 'slow': slow, **metrics})
    ranking = pd.DataFrame(records).sort_values(['sharpe', 'fast', 'slow'], ascending=[False, True, True]).reset_index(drop=True)
    winner = ranking.iloc[0]
    return {'fast': int(winner['fast']), 'slow': int(winner['slow'])}, ranking


In [ ]:
data = synthetic_prices(SEED) if CSV_PATH is None else validate_prices(pd.read_csv(CSV_PATH))
if not 0.5 <= TRAIN_FRACTION <= 0.8:
    raise ValueError('Training fraction must be between 0.5 and 0.8')
split = int(len(data) * TRAIN_FRACTION)
DATA_LABEL = 'Seeded synthetic prices' if CSV_PATH is None else 'User-supplied daily CSV'
print(DATA_LABEL, '| rows:', len(data), '| SHA-256:', fingerprint(data))
print('Train through:', data.iloc[split-1]['date'].date(), '| Test starts:', data.iloc[split]['date'].date())
display(data.head())


## Select on training data only
All candidates use the same warmup cutoff. Select the highest training Sharpe; ties prefer the smaller windows. This is a demonstration search, not a statistical significance test. Do not select using the held-out table below.


In [ ]:
chosen, training_ranking = select_on_training(data, split, CANDIDATES, COST_BPS)
display(training_ranking)
print('Frozen parameters:', chosen)


## Evaluate the frozen rule once
The test can use earlier prices to calculate its first signal, but no future prices influence that signal. The position resets to cash at the test boundary, so its first entry incurs a cost. Results are for the test period only.


In [ ]:
test_path, test_metrics = backtest(data, **chosen, cost_bps=COST_BPS, start=split)
display(pd.Series(test_metrics, name='held_out_test'))
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(test_path.date, test_path.equity, color='#c5ee89', label='Frozen strategy, net of costs')
axes[0].plot(test_path.date, test_path.benchmark, color='#91a3a7', linestyle='--', label='Buy & hold, entry cost')
axes[0].set(title='Asnya Research · Held-out equity · ' + DATA_LABEL, ylabel='Equity, start = 100')
axes[0].legend()
axes[1].fill_between(test_path.date, 100 * test_path.drawdown, 0, color='#efa69c', alpha=0.6)
axes[1].set(ylabel='Drawdown %', xlabel='Date')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'held_out_equity.png', dpi=150)
plt.show()


## Export the evidence
The manifest captures parameters, input hash, split, engine and installed package versions. It does not prove statistical significance. Preserve the original CSV alongside this manifest for real-data reproduction. No automatic external data request is made.


In [ ]:
manifest = {
    'engine': ENGINE_VERSION, 'data_label': DATA_LABEL, 'seed': SEED if CSV_PATH is None else None,
    'data_sha256': fingerprint(data), 'rows': len(data), 'split_index': split,
    'train_end': str(data.iloc[split-1]['date'].date()), 'test_start': str(data.iloc[split]['date'].date()),
    'candidates': CANDIDATES, 'selected': chosen, 'cost_bps': COST_BPS,
    'test_metrics': test_metrics,
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'matplotlib': matplotlib.__version__}
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False))
training_ranking.to_csv(OUTPUT_DIR / 'training_ranking.csv', index=False)
test_path.to_csv(OUTPUT_DIR / 'held_out_returns.csv', index=False)
print('Exported manifest, ranking, returns and chart to', OUTPUT_DIR.resolve())
assert np.isfinite(test_path.equity).all()
assert test_path.date.iloc[1] == data.date.iloc[split]
